In [1]:
from costs_concrete import *
from math import comb

# Calculating BIM server storage with rebalancing

The below function takes as input n (DB size in records) and CCmax (an upper bound on the download), and finds the smallest possible storage (in records) that can be obtained using the BIM construction together with rebalancing.

It does this by searching over values of m and D and determining the number of records r that need to be combined into one "mega-record" for rebalancing.

In [2]:
# n: db size in records
# CCmax: upper bound on download in records
def get_bim_costs(n, CCmax):
    best_storage = float('inf')
    for m in range(2, maxm):
        for D in range(1, min(m+1, maxD)):
            # number of mega-records
            n2 = db_size(D, 1, m, 2, homog=True)

            # number of records per mega-record
            r = int(np.ceil(n/n2))

            # download in records
            CC = r * server_online_time(D, 1, m, 2, 2, homog=True, use_finite_differences=False)

            if CC <= CCmax:
                test_storage = r * preprocessing_cost(D, 1, m, 2, 2, homog=True, use_finite_differences=False)

                if test_storage < best_storage:
                    best_storage = test_storage
                    param_dict = {'m': m, 'D': D, 'r': r}

    return best_storage, param_dict

This example is for the bottom row of the right subtable of table 1

In [24]:
D = 17
m = 40
l = 1 # length of a record in bytes

DB size, in GB

In [25]:
n = db_size(D, 1, m, 2, homog=True) * l
assert n == comb(m, D) * l
print(n/pow(1024, 3)) # GB

82.6384674757719


Our server storage. For all examples in our table, this is 1 TB.

In [26]:
our_storage = preprocessing_cost(D, 1, m, 2, 2, homog=True, use_finite_differences=True) * l
assert our_storage == pow(2, m) * l
print(our_storage/pow(1024, 4)) # TB

1.0


In [27]:
our_CC = server_online_time(D, 1, m, 2, 2, use_finite_differences=True)
print(our_CC/pow(1024, 2)) # MB

95.50735855102539


Calculating BIM server storage, in TB

In [28]:
bim_storage, params = get_bim_costs(n, our_CC)
params

{'m': 34, 'D': 9, 'r': 1692}

In [29]:
blowup = bim_storage/our_storage

In [30]:
blowup/1e5

14.0002425